In [21]:
import qiskit 
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile 

# Get the q0,q1,q2 states
cr = ClassicalRegister(3, 'c')
qr = QuantumRegister(3, 'q') 
qc = QuantumCircuit(qr, cr)

# prepare the qubit to be teleported 
# Apply X gate to the first qubit to prepare the state |1>
qc.x(0)
qc.barrier()

# Create a Bell pair between qubits 1 and 2
qc.h(1)
qc.cx(1, 2)
qc.barrier()

# Create the Bell state between qubits 0 and 1
qc.cx(0, 1)
qc.h(0)
qc.barrier()

# Measure the qubits 0 and 1 and store the results in classical bits 0 and 1
qc.measure(0, 0)  
qc.measure(1, 1)

# Apply conditional operations based on the measurement results
with qc.if_test((qc.clbits[1], 1)):
    qc.x(2)

with qc.if_test((qc.clbits[0], 1)):
    qc.z(2)

# Measure the teleported qubit and store the result in classical bit 2
qc.measure(2, 2)
print(qc.draw('text'))

# Simulate the circuit
from qiskit_aer import AerSimulator
from qiskit import transpile
# Simulate the circuit
simulator = AerSimulator()
compiled_circuit = transpile(qc, simulator)
result = simulator.run(compiled_circuit, shots=1024).result()
counts = result.get_counts(qc)
print("\nCounts:", counts)


     ┌───┐ ░            ░      ┌───┐ ░ ┌─┐                             »
q_0: ┤ X ├─░────────────░───■──┤ H ├─░─┤M├─────────────────────────────»
     └───┘ ░ ┌───┐      ░ ┌─┴─┐└───┘ ░ └╥┘┌─┐                          »
q_1: ──────░─┤ H ├──■───░─┤ X ├──────░──╫─┤M├──────────────────────────»
           ░ └───┘┌─┴─┐ ░ └───┘      ░  ║ └╥┘  ┌──────  ┌───┐ ───────┐ »
q_2: ──────░──────┤ X ├─░────────────░──╫──╫───┤ If-0  ─┤ X ├  End-0 ├─»
           ░      └───┘ ░            ░  ║  ║   └──╥───  └───┘ ───────┘ »
                                        ║  ║ ┌────╨────┐               »
c: 3/═══════════════════════════════════╩══╩═╡ c_1=0x1 ╞═══════════════»
                                        0  1 └─────────┘               »
«                                  
«q_0: ─────────────────────────────
«                                  
«q_1: ─────────────────────────────
«       ┌──────  ┌───┐ ───────┐ ┌─┐
«q_2: ──┤ If-0  ─┤ Z ├  End-0 ├─┤M├
«       └──╥───  └───┘ ───────┘ └╥┘
«     ┌────╨────┐ 